Classic & Baseline Methods
- Frequentist Hypothesis Testing (e.g., t-tests, z-tests)
- Chi-square tests
- ANOVA

Advanced Statistical Methods
- Linear Approximation (Delta Method)
- Sequential Testing (Sequential Probability Ratio Test - SPRT)
- Multi-Armed Bandit (MAB)
- Bayesian A/B Testing

Variance Reduction Techniques
- CUPED (Controlled-Experiment Using Pre-Experiment Data)
- Multiple CUPED
- CUPAC (Controlled-Experiment Using Predictions as Covariates)
- CUPIT (Controlled-experiment Using Predictions of Individual Treatment effects)

Bayesian and Hybrid Methods
- Bayesian Hierarchical Models
- Empirical Bayes
- Thompson Sampling
- Contextual Bandits

Non-standard & Emerging Techniques
- Sequential Monitoring and Early Stopping

- Heterogeneous Treatment Effect (HTE) Estimation

Causal Inference Approaches
- Matching and Propensity Score Matching
- Synthetic Controls
- Instrumental Variables (IV)
- Meta-Analysis and Multi-level Experimentation
- Factorial Designs

Adaptive Experimentation
- Machine Learning & Predictive Methods
- Machine Learning-based Predictive Experimentation
- Personalized A/B Testing (Individual-level Experimentation)
- Multi-variate Testing

Experimentation Platforms & Frameworks
- Google's Exp Platform (including methods like Cupid, multiple Cupid)
- Netflix’s experimentation framework
- Meta's (Facebook’s) Vortex
- Airbnb’s experimentation platform
- Microsoft's ExP (Controlled Experiments)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

L_train = 32768   # Training context length (32K)
L_test = 131072   # Inference context length (128K)

positions = np.arange(0, L_test, 512)
pos_interp = positions * (L_train / L_test)

fig, ax = plt.subplots(figsize=(8, 5))
line, = ax.plot([], [], lw=2)
ax.set_xlim(0, L_test)
ax.set_ylim(0, L_train)
ax.set_xlabel("Original Position (L_test)")
ax.set_ylabel("Interpolated Position (L_train)")
ax.set_title("RoPE Position Interpolation")

def init():
    line.set_data([], [])
    return line,

def animate(i):
    x = positions[:i]
    y = pos_interp[:i]
    line.set_data(x, y)
    return line,

ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(positions), interval=50, blit=True)
ani.save("rope_interpolation.gif", writer=animation.PillowWriter(fps=20))


ModuleNotFoundError: No module named 'matplotlib'

# Variance reduction
---
The variance in target metric is not totally random, it always contains some part of systematic spread. This is important beacuse it means we can exclude it. If not done, it increases the variance

__CUPED__<br>
CUPED = Controlled-experiment Using Pre-Experiment Data

Idea: let's use a normalized version of target metric - replace it with a residual $Y_{res} = Y - \hat{Y}$ where $\hat{Y}$ is a regression of $Y$ on $X$ and represents the deterministic part of the variance of Y $$Y_{res} = Y - \hat{Y} = (X-\mathbb{E}{X}) \cdot \frac{Cov(X,Y)}{Var(x)}$$

__Multiple CUPED__<br>
The same approach but let's regress on multiple features $X = (X_1, X_2 ... X_N)$

__CUPAC__<br>
CUPAC = Controlled-experiment Using Predictions As Covariates. Instead of solving regression of $Y$ on $X$ use an arbitrary predictive model

__CUPIT__<br>
CUPIT = Controlled-experiment Using Predictions of Individual Treatment effects<br>Instead of solving regression of $Y$ on $X$ use a personalized effect prediction

---

Why CUPED reduces variance

Let's show that normalized version of the metric<br>
$
Y^* = Y - \theta (X - \bar{X}) \quad \mathrm{where} \,
\theta = \frac{\text{Cov}(X, Y)}{\text{Var}(X)}
$

has smaller variance then the original<br>
$\text{Var}(Y^*) < \text{Var}(Y)$

Expand<br>
$
\text{Var}(Y^*) = \text{Var}(Y - \theta (X - \bar{X})) = \text{Var}(Y - \theta X)
$

Variance of the sum<br>
$
\text{Var}(Y - \theta X) = \text{Var}(Y) + \theta^2 \text{Var}(X) - 2\theta \text{Cov}(X, Y)
$

Plug the \theta<br>
$
\text{Var}(Y^*) = \text{Var}(Y) + \left( \frac{\text{Cov}(X, Y)^2}{\text{Var}(X)^2} \right) \text{Var}(X) - 2 \left( \frac{\text{Cov}(X, Y)}{\text{Var}(X)} \right) \text{Cov}(X, Y)
$

Thus we get<br>
$
\text{Var}(Y^*) = \text{Var}(Y) - \frac{\text{Cov}(X, Y)^2}{\text{Var}(X)}
$

This proves that
$$
\text{Var}(Y^*) < \text{Var}(Y) \quad \text{if } \text{Cov}(X, Y) \ne 0
$$